# ComfyUI v0.38.2 + Pixal3D / TRELLIS.2 (V2 mesh repair)

Run **ComfyUI** on a Colab GPU and open it through Colab's built-in proxy.
No external tunnel, SSH key, or extra credential.
**Mesh Repair Lite** repairs watertight and non-manifold defects while keeping the generated surface.
Each connected component is repaired on its own, so wings, hinges, and trim are not dropped.

| Item | Value |
|------|--------|
| ComfyUI | `v0.38.2` |
| Access | Colab proxy (`google.colab.kernel.proxyPort`) |
| Template | Pixal3D & TRELLIS.2 Image to Model |
| Repair | weld → GPU QEM → per-component pymeshfix → optional inner shell → normals |

Use an **A100** runtime. Run the cells from the top. The repair node is cloned from this GitHub repo. Google Drive is not used.


## 1. Check the GPU


In [ ]:
import torch
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print("vram_gb", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("torch", torch.__version__)


## 2. Where outputs go

GLB files stay in `/content/ComfyUI/output` on the Colab disk.
This notebook does not mount Google Drive.


In [ ]:
from pathlib import Path

OUT = Path("/content/ComfyUI/output")
OUT.mkdir(parents=True, exist_ok=True)
print("outputs:", OUT)


## 3. Install ComfyUI v0.38.2

Keep Colab's CUDA torch. Install everything else from `requirements.txt`.


In [ ]:
import os, subprocess, sys
from pathlib import Path

COMFY_TAG = "v0.38.2"
ROOT = Path("/content/ComfyUI")
REPO = "https://github.com/Comfy-Org/ComfyUI.git"

if not (ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", COMFY_TAG, "--depth", "1", REPO, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "fetch", "--tags", "--depth", "1", "origin", f"refs/tags/{COMFY_TAG}:refs/tags/{COMFY_TAG}"], check=False)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "--force", COMFY_TAG], check=True)

os.chdir(ROOT)
req = (ROOT / "requirements.txt").read_text(encoding="utf-8")
keep = []
for line in req.splitlines():
    s = line.strip()
    if not s or s.startswith("#"):
        keep.append(line)
        continue
    name = s.split("==")[0].split(">=")[0].split("~=")[0].split("[")[0].strip().lower()
    if name in {"torch", "torchvision", "torchaudio"}:
        continue
    keep.append(line)
tmp = Path("/tmp/comfy_req_notorch.txt")
tmp.write_text("\n".join(keep) + "\n", encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(tmp)], check=True)
print("ready", ROOT, "tag", COMFY_TAG)


## 3b. Optional SD1.5 smoke test

Skip this if you only need the UI. Install it if you will queue a test (about 2GB).


In [ ]:
#@title Install SD1.5
DOWNLOAD_SD15 = False  #@param {type:"boolean"}
from pathlib import Path
import urllib.request

dst = Path("/content/ComfyUI/models/checkpoints/v1-5-pruned-emaonly-fp16.safetensors")
url = "https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors"
if DOWNLOAD_SD15:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_file() and dst.stat().st_size > 1_000_000_000:
        print("already", dst, dst.stat().st_size)
    else:
        print("download", url)
        urllib.request.urlretrieve(url, dst)
        print("saved", dst, dst.stat().st_size)
else:
    print("skip")


## 3.5 Download TRELLIS.2 / Pixal3D models on Colab

A download started from the proxied UI often saves to your own PC.
This cell writes straight into `/content/ComfyUI/models/` on the VM. Run it before the launch cell.
The first run is tens of GB. Files already present are skipped.


In [ ]:
# Download TRELLIS.2 / Pixal3D weights onto the Colab VM
from pathlib import Path
import shutil

try:
    from huggingface_hub import hf_hub_download
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
    from huggingface_hub import hf_hub_download

ROOT = Path('/content/ComfyUI/models')
jobs = [
    ('Comfy-Org/TRELLIS.2', 'diffusion_models/trellis_2_int8_convrot.safetensors', 'diffusion_models'),
    ('Comfy-Org/TRELLIS.2', 'vae/trellis_2_shape_vae_bf16.safetensors', 'vae'),
    ('Comfy-Org/TRELLIS.2', 'vae/trellis_2_texture_vae_bf16.safetensors', 'vae'),
    ('Comfy-Org/Pixal3D', 'clip_vision/dino_v3_L_naf_fp32.safetensors', 'clip_vision'),
    ('Comfy-Org/Pixal3D', 'diffusion_models/pixal3d_int8_convrot.safetensors', 'diffusion_models'),
    ('Comfy-Org/MoGe', 'geometry_estimation/moge_2_vitl_normal_fp16.safetensors', 'geometry_estimation'),
    ('Comfy-Org/BiRefNet', 'background_removal/birefnet.safetensors', 'background_removal'),
]

for repo, repo_path, sub in jobs:
    dest_dir = ROOT / sub
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / Path(repo_path).name
    if dest.is_file() and dest.stat().st_size > 1_000_000:
        print('skip', dest.name, f'{dest.stat().st_size/1e9:.2f}GB')
        continue
    print('downloading', repo_path, '...')
    cached = hf_hub_download(repo_id=repo, filename=repo_path)
    shutil.copy2(cached, dest)
    print('OK', dest, f'{dest.stat().st_size/1e9:.2f}GB')

print('done. Reload the browser if ComfyUI still lists missing models')


## 3.7 Install Mesh Repair Lite

Topology repair node. In the workflow it sits after
`RemeshMesh` (udf, original settings) and `DecimateMesh` (700k / midpoint).
Inside: weld vertices, GPU QEM, **pymeshfix on each connected component**,
optional inner-shell removal, then normal unify.

The node is cloned from this GitHub repository. Run this cell **before** the launch cell.
ComfyUI loads custom nodes at startup, so a later run will not be picked up.


In [ ]:
#@title Install Mesh Repair Lite (run before launch)
REPO_URL = "https://github.com/koba4061/trellis2-pixal3d-comfyui-mesh-repair.git"  #@param {type:"string"}
import shutil, subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "trimesh", "scipy", "pymeshfix", "pymeshlab", "fast-simplification"], check=False)

src_root = Path("/tmp/trellis2-pixal3d-mesh-repair")
if src_root.exists():
    shutil.rmtree(src_root)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(src_root)], check=True)
src = src_root / "custom_nodes" / "mesh_repair_lite"
dst = Path("/content/ComfyUI/custom_nodes/mesh_repair_lite")
if not src.is_dir():
    raise SystemExit(f"MISSING: {src}")
dst.parent.mkdir(parents=True, exist_ok=True)
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print("copied", src, "->", dst)
print("files:", sorted(p.name for p in dst.iterdir()))


## 4. Launch and open the built-in proxy

`google.colab.kernel.proxyPort` forwards a port on this runtime.
No ngrok, Cloudflare, or SSH key.

The printed URL works only in a browser signed into the **same Google account** that has this notebook open. It is not a public link. It stops when the runtime stops. Leave this cell running.


In [ ]:
#@title 4. Start ComfyUI and print the Colab proxy URL
import os, subprocess, sys, time, urllib.request, threading
from pathlib import Path

COMFY_PORT = 8188
ROOT = Path("/content/ComfyUI")
if not (ROOT / "main.py").is_file():
    raise RuntimeError("Run the install cell first")
os.chdir(ROOT)
subprocess.run(f"pkill -f 'python.*main.py' 2>/dev/null || true; fuser -k {COMFY_PORT}/tcp 2>/dev/null || true", shell=True)
time.sleep(2)

def _drain(pr):
    for line in pr.stdout:
        print(line, end=""); sys.stdout.flush()

log = Path("/tmp/comfyui.log")
p = subprocess.Popen(
    ["/bin/bash", "-lc", f"{sys.executable} -u main.py --listen 127.0.0.1 --port {COMFY_PORT} "
     f"--enable-cors-header '*' --dont-print-server --preview-method auto 2>&1 | tee {log}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
threading.Thread(target=_drain, args=(p,), daemon=True).start()

t0 = time.time()
while time.time() - t0 < 180:
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/", timeout=3); break
    except Exception:
        if p.poll() is not None:
            raise SystemExit(f"ComfyUI failed to start. See {log}")
        time.sleep(1)

from google.colab.output import eval_js
print("\nbrowser:", eval_js(f"google.colab.kernel.proxyPort({COMFY_PORT})"))
print("(only the Google account that has this notebook open)")
p.wait()
